# Food to Recipe Generation using Qwen2.5-VL-3B

## Project Overview

This notebook implements a deep learning system that generates cooking recipes from images of food using **Qwen2.5-VL-3B-Instruct** as a native vision-language model.

### Key Features:
- ✅ **Native Vision-Language Model**: Qwen2.5-VL handles both vision and language in one model
- ✅ **Instruction-Tuned**: Better at following prompts and generating structured recipes
- ✅ **Multi-GPU Support**: Optimized for training on multiple GPUs with DeepSpeed ZeRO
- ✅ **Memory Efficient**: Gradient checkpointing and FP16 precision for efficient training

### Approach
1. Load and preprocess food image-recipe dataset
2. Fine-tune Qwen2.5-VL-3B on image-recipe pairs
3. Generate recipes from food images
4. Evaluate recipe quality using various metrics

## 1. Installation and Imports

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import zipfile
import os
import re
import random
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Metrics libraries
try:
    from rouge_score import rouge_scorer
    ROUGE_AVAILABLE = True
except ImportError:
    print("Installing rouge-score...")
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "rouge-score"])
    from rouge_score import rouge_scorer
    ROUGE_AVAILABLE = True

try:
    import sacrebleu
    BLEU_AVAILABLE = True
except ImportError:
    print("Installing sacrebleu...")
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "sacrebleu"])
    import sacrebleu
    BLEU_AVAILABLE = True

# Set PyTorch memory management
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Hugging Face Libraries
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    TrainingArguments,
    Trainer
)
from qwen_vl_utils import process_vision_info

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully!")

✓ Libraries imported successfully!


## 2. Device Configuration

In [ ]:
# Configure device and GPU settings
if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    device = torch.device("cuda")
    use_multi_gpu = num_gpus > 1
    print(f"Found {num_gpus} GPU(s)")
    for i in range(num_gpus):
        gpu_name = torch.cuda.get_device_name(i)
        gpu_memory = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i}: {gpu_name} ({gpu_memory:.1f} GB)")
    if use_multi_gpu:
        print(f"✓ Multi-GPU training enabled with {num_gpus} GPUs")
else:
    device = torch.device("cpu")
    num_gpus = 0
    use_multi_gpu = False
    print("⚠️ Using CPU (No GPU available - training will be very slow)")

print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    # Enable optimizations
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("✓ CUDA optimizations enabled")

Found 2 GPU(s)
  GPU 0: NVIDIA GeForce RTX 3090 (25.3 GB)
  GPU 1: NVIDIA GeForce RTX 3090 (25.3 GB)
✓ Multi-GPU training enabled with 2 GPUs

PyTorch version: 2.9.0+cu128
CUDA available: True
CUDA version: 12.8
✓ CUDA optimizations enabled


## 3. Dataset Loading

In [ ]:
# Extract dataset if zipped
zip_file = "Food Ingredients and Recipe Dataset with Image Name Mapping.csv.zip"
target_csv = "Food Ingredients and Recipe Dataset with Image Name Mapping.csv"

if os.path.exists(zip_file) and not os.path.exists(target_csv):
    print(f"Extracting {zip_file}...")
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(".")
    print(f"✓ Extracted successfully!")

# Load dataset
if os.path.exists(target_csv):
    main_dataset = pd.read_csv(target_csv)
    print(f"✓ Loaded {len(main_dataset)} recipes")
    print(f"\nDataset columns: {list(main_dataset.columns)}")
    print(f"\nSample recipe:")
    print(f"  Title: {main_dataset.iloc[0]['Title']}")
else:
    raise FileNotFoundError(f"Dataset '{target_csv}' not found!")

✓ Loaded 13501 recipes

Dataset columns: ['Unnamed: 0', 'Title', 'Ingredients', 'Instructions', 'Image_Name', 'Cleaned_Ingredients']

Sample recipe:
  Title: Miso-Butter Roast Chicken With Acorn Squash Panzanella


## 4. Image Path Resolution

In [ ]:
def find_image_file(image_name, search_dirs=['food_images/Food Images', 'food_images', '.']):
    """Find image file by name in common directories."""
    extensions = ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']
    
    for directory in search_dirs:
        if not os.path.exists(directory):
            continue
        
        for ext in extensions:
            path = os.path.join(directory, image_name + ext)
            if os.path.exists(path) and os.path.isfile(path):
                return path
    
    return None

# Process dataset and find images
processed_data = []

print("Processing dataset and finding images...")
for idx, row in main_dataset.iterrows():
    image_name = row['Image_Name']
    image_path = find_image_file(image_name)
    
    if image_path is None:
        continue
    
    # Format recipe for Qwen
    title = row['Title']
    ingredients = row['Ingredients']
    instructions = row['Instructions']
    
    recipe_output = f"""RECIPE: {title}

INGREDIENTS:
{ingredients}

INSTRUCTIONS:
{instructions}"""
    
    processed_data.append({
        'image_path': image_path,
        'title': title,
        'ingredients': ingredients,
        'instructions': instructions,
        'recipe': recipe_output
    })

print(f"✓ Processed {len(processed_data)} samples with valid images out of {len(main_dataset)} total")
if processed_data:
    print(f"\nSample: {processed_data[0]['title']}")
    print(f"  Image: {processed_data[0]['image_path']}")

Processing dataset and finding images...
✓ Processed 0 samples with valid images out of 13501 total


## 5. Load Qwen2.5-VL-3B Model

In [ ]:
# Load Qwen2.5-VL-3B-Instruct model
print("Loading Qwen2.5-VL-3B-Instruct...")
print("This may take a few minutes...")

model_name = "Qwen/Qwen2.5-VL-3B-Instruct"

# Clear GPU cache before loading
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Load model - Single GPU mode
# Load directly to GPU 0 for single GPU training
qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_name,
    dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="auto",  # Auto device placement (will use GPU 0)
)

# Load processor
qwen_processor = AutoProcessor.from_pretrained(model_name)

# Enable gradient checkpointing to save memory during training
qwen_model.gradient_checkpointing_enable()

print("✓ Qwen2.5-VL loaded successfully!")
print(f"  Model parameters: {sum(p.numel() for p in qwen_model.parameters()):,}")
print(f"  Trainable parameters: {sum(p.numel() for p in qwen_model.parameters() if p.requires_grad):,}")
model_device = next(qwen_model.parameters()).device
print(f"  Model device: {model_device}")

Loading Qwen2.5-VL-3B-Instruct...
This may take a few minutes...


Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.23s/it]
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


✓ Qwen2.5-VL loaded successfully!
  Model parameters: 3,754,622,976
  Trainable parameters: 3,754,622,976
  Model device: cuda:0


## 6. Dataset Class Definition

In [ ]:
class QwenRecipeDataset(Dataset):
    """Dataset class for Qwen recipe generation."""
    
    def __init__(self, data, processor, max_length=2048):
        self.data = data
        self.processor = processor
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Load image
        image = Image.open(item['image_path']).convert('RGB')
        
        # Create prompt
        prompt_text = "Generate a detailed cooking recipe from this food image."
        
        # Create messages for Qwen
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt_text}
                ]
            },
            {
                "role": "assistant",
                "content": item['recipe']
            }
        ]
        
        # Apply chat template
        text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        
        # CRITICAL: Pass images directly - processor will compute image_grid_thw automatically
        # Do NOT use vision_infos - let the processor handle everything
        inputs = self.processor(
            text=[text],
            images=[image],  # Pass PIL Image directly - processor computes grid_thw
            padding=True,
            return_tensors="pt"
        )
        
        # Move to CPU and remove batch dimension (batch_size=1)
        result = {}
        for k, v in inputs.items():
            if k == 'image_grid_thw':
                # image_grid_thw is a numpy array or tensor from processor
                # For single image: shape should be (1, 3) -> extract (3,) -> convert to tuple
                if isinstance(v, torch.Tensor):
                    v_np = v.cpu().numpy() if v.is_cuda else v.numpy()
                elif hasattr(v, '__array__'):
                    v_np = np.array(v)
                else:
                    v_np = v
                
                # Handle shape: (1, 3) -> extract first row -> tuple of 3
                if isinstance(v_np, np.ndarray):
                    if v_np.ndim == 2 and v_np.shape[0] == 1:
                        # Single image: (1, 3) -> (3,)
                        result[k] = tuple(v_np[0].astype(int))
                    elif v_np.ndim == 1 and v_np.shape[0] == 3:
                        # Already (3,)
                        result[k] = tuple(v_np.astype(int))
                    else:
                        # Fallback: take first element if possible
                        result[k] = tuple(v_np.flatten()[:3].astype(int)) if v_np.size >= 3 else (1, 1, 1)
                elif isinstance(v_np, (list, tuple)):
                    # Convert list to tuple
                    if len(v_np) == 1 and isinstance(v_np[0], (list, tuple, np.ndarray)):
                        result[k] = tuple(v_np[0][:3]) if len(v_np[0]) >= 3 else (1, 1, 1)
                    elif len(v_np) >= 3:
                        result[k] = tuple(v_np[:3])
                    else:
                        result[k] = (1, 1, 1)
                else:
                    result[k] = (1, 1, 1)  # Fallback
            elif isinstance(v, torch.Tensor):
                # Remove batch dimension if present (batch_size=1)
                if v.dim() > 0 and v.shape[0] == 1:
                    result[k] = v.squeeze(0)
                else:
                    result[k] = v
            else:
                result[k] = v
        
        # Ensure image_grid_thw exists and is valid tuple of 3 positive integers
        if 'image_grid_thw' not in result:
            result['image_grid_thw'] = (1, 1, 1)
        else:
            grid_thw = result['image_grid_thw']
            # Validate: must be tuple of 3 positive integers
            if not isinstance(grid_thw, tuple) or len(grid_thw) != 3:
                result['image_grid_thw'] = (1, 1, 1)
            elif any(not isinstance(x, (int, np.integer)) or x <= 0 for x in grid_thw):
                result['image_grid_thw'] = (1, 1, 1)
            else:
                # Ensure all are positive integers
                result['image_grid_thw'] = tuple(int(max(1, x)) for x in grid_thw)
        
        # Extract labels for training
        labels = result['input_ids'].clone()
        result['labels'] = labels
        
        return result

## 7. Create Train/Test Split

In [ ]:
# Create dataset splits
random.seed(42)
random.shuffle(processed_data)

# Limit dataset size for faster iteration (adjust as needed)
max_samples = 1000  # Use up to 1000 samples for training
dataset_subset = processed_data[:max_samples]

train_size = int(0.8 * len(dataset_subset))
test_size = len(dataset_subset) - train_size

train_data = dataset_subset[:train_size]
test_data = dataset_subset[train_size:]

print(f"Dataset split:")
print(f"  Training samples: {len(train_data)}")
print(f"  Test samples: {len(test_data)}")

# Create PyTorch datasets
train_dataset = QwenRecipeDataset(train_data, qwen_processor)
test_dataset = QwenRecipeDataset(test_data, qwen_processor)

Dataset split:
  Training samples: 0
  Test samples: 0


## 8. Training Configuration

In [ ]:
# Training arguments optimized for memory efficiency
per_device_batch = 1
gradient_accum = 64  # Adjust based on your GPU memory

effective_batch_size = per_device_batch * gradient_accum
if use_multi_gpu:
    effective_batch_size *= num_gpus

training_args = TrainingArguments(
    output_dir="./qwen_recipe_output",
    num_train_epochs=3,
    per_device_train_batch_size=per_device_batch,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=gradient_accum,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    max_grad_norm=1.0,
    logging_dir="./logs",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    fp16=True,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    report_to="none",
    gradient_checkpointing=True,
    optim="adamw_torch",
    weight_decay=0.01,
    torch_empty_cache_steps=5,
    dataloader_drop_last=True,
    ddp_find_unused_parameters=False,
)

print("✓ Training configuration ready!")
print(f"  GPUs: {num_gpus if use_multi_gpu else 1}")
print(f"  Batch per GPU: {per_device_batch}")
print(f"  Gradient accumulation: {gradient_accum}")
print(f"  Effective batch size: {effective_batch_size}")

✓ Training configuration ready!
  GPUs: 2
  Batch per GPU: 1
  Gradient accumulation: 64
  Effective batch size: 128


## 9. Optional: DeepSpeed Configuration (for Multi-GPU)

**Note**: If you see linker errors about `-laio` when importing DeepSpeed, these are **non-critical warnings**. DeepSpeed will work fine without the `libaio` library - you'll just miss some optional I/O optimizations. To install the library (optional):
```bash
sudo apt-get install libaio-dev
```

In [ ]:
# DeepSpeed ZeRO configuration for memory efficiency (optional)
USE_DEEPSPEED = True  # Set to False to disable

if USE_DEEPSPEED and use_multi_gpu:
    try:
        # Import DeepSpeed (suppress linker warnings if they occur)
        import sys
        import io
        from contextlib import redirect_stderr, redirect_stdout
        
        # Suppress both stdout/stderr during import to hide non-critical linker warnings
        with redirect_stderr(io.StringIO()), redirect_stdout(io.StringIO()):
            import deepspeed
        print("✓ DeepSpeed is installed")
    except ImportError:
        print("⚠️ DeepSpeed not installed. Installing...")
        import subprocess
        import sys
        # Suppress linker warnings during installation
        import os
        import io
        from contextlib import redirect_stderr
        
        # Install DeepSpeed (linker warnings about libaio are non-critical)
        with redirect_stderr(io.StringIO()):
            result = subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", "deepspeed"],
                capture_output=True,
                text=True
            )
        import deepspeed
        print("✓ DeepSpeed installed")
        print("  Note: Linker warnings about libaio are non-critical.")
        print("  DeepSpeed will work, but install libaio-dev for full I/O optimizations:")
        print("    sudo apt-get install libaio-dev")
    
    # Create DeepSpeed config
    deepspeed_config = {
        "zero_optimization": {
            "stage": 2,  # Stage 2 is usually sufficient
            "offload_optimizer": {"device": "cpu", "pin_memory": True},
            "overlap_comm": True,
            "contiguous_gradients": True,
        },
        "gradient_accumulation_steps": training_args.gradient_accumulation_steps,
        "gradient_clipping": training_args.max_grad_norm,
        "train_micro_batch_size_per_gpu": training_args.per_device_train_batch_size,
        "train_batch_size": "auto",
        "fp16": {"enabled": True},
        "steps_per_print": 10,
    }
    
    # Save config
    config_path = "ds_config.json"
    with open(config_path, "w") as f:
        json.dump(deepspeed_config, f, indent=2)
    
    # Set in training args
    training_args.deepspeed = config_path
    
    print(f"\n✓ DeepSpeed ZeRO Stage 2 enabled")
    print(f"  Config saved to: {config_path}")
    print(f"  Note: Linker warnings about '-laio' are non-critical - DeepSpeed will work fine")
else:
    print("⚠️ DeepSpeed disabled or not needed (single GPU)")

✓ DeepSpeed is installed

✓ DeepSpeed ZeRO Stage 2 enabled
  Config saved to: ds_config.json
  Note: Linker warnings about '-laio' are non-critical - DeepSpeed will work fine


## 10. Data Collator

In [ ]:
from typing import Dict, List

class QwenDataCollator:
    """Custom data collator for Qwen multimodal inputs."""
    
    def __init__(self, processor):
        self.processor = processor
    
    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        batch = {}
        
        # Stack image inputs
        if 'pixel_values' in features[0]:
            batch['pixel_values'] = torch.stack([f['pixel_values'] for f in features])
        
        if 'pixel_values_vl' in features[0]:
            batch['pixel_values_vl'] = torch.stack([f['pixel_values_vl'] for f in features])
        
        # CRITICAL: Preserve image_grid_thw metadata
        # image_grid_thw should be a tensor of shape (batch_size, 3) for the batch
        # Each feature has image_grid_thw as a tuple (t, h, w) for its single image
        if 'image_grid_thw' in features[0]:
            image_grid_thw_list = []
            for f in features:
                grid_thw = f.get('image_grid_thw', None)
                if grid_thw is not None:
                    # Ensure it's a tuple of 3 positive integers
                    if isinstance(grid_thw, (list, tuple)):
                        if len(grid_thw) == 3 and all(isinstance(x, (int, np.integer)) for x in grid_thw):
                            # Validate all are positive
                            t, h, w = grid_thw
                            if t > 0 and h > 0 and w > 0:
                                image_grid_thw_list.append([int(t), int(h), int(w)])
                            else:
                                image_grid_thw_list.append([1, 1, 1])
                        else:
                            # Invalid format - use default
                            image_grid_thw_list.append([1, 1, 1])
                    else:
                        # Not a tuple/list - use default
                        image_grid_thw_list.append([1, 1, 1])
                else:
                    # Missing - use default
                    image_grid_thw_list.append([1, 1, 1])
            
            # Convert to tensor: shape (batch_size, 3)
            # CRITICAL: Must be LongTensor with positive integers
            batch['image_grid_thw'] = torch.tensor(image_grid_thw_list, dtype=torch.long)
        
        # Handle input_ids with padding
        if 'input_ids' in features[0]:
            max_length = max(len(f['input_ids']) for f in features)
            max_length = min(max_length, 2048)  # Cap at 2048
            
            pad_token_id = self.processor.tokenizer.pad_token_id
            if pad_token_id is None:
                pad_token_id = self.processor.tokenizer.eos_token_id
            
            input_ids = []
            attention_mask = []
            labels = []
            
            for f in features:
                ids = f['input_ids'][:max_length]
                padding_length = max_length - len(ids)
                
                ids = torch.cat([ids, torch.full((padding_length,), pad_token_id, dtype=ids.dtype)])
                mask = torch.cat([torch.ones(len(f['input_ids'][:max_length]), dtype=torch.bool),
                                torch.zeros(padding_length, dtype=torch.bool)])
                
                label = f['labels'][:max_length] if 'labels' in f else ids.clone()
                # Mask padding tokens in labels
                label_padding = torch.full((padding_length,), -100, dtype=label.dtype)
                label = torch.cat([label, label_padding])
                
                input_ids.append(ids)
                attention_mask.append(mask)
                labels.append(label)
            
            batch['input_ids'] = torch.stack(input_ids)
            batch['attention_mask'] = torch.stack(attention_mask)
            batch['labels'] = torch.stack(labels)
        
        return batch

data_collator = QwenDataCollator(qwen_processor)
print("✓ Data collator created")

✓ Data collator created


## 11. Clear GPU Memory Before Training


In [ ]:
# Clear GPU memory before training (Single GPU mode)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(0)
    torch.cuda.synchronize(0)
    print("✓ GPU memory cleared")
    
    # Print current memory usage
    allocated = torch.cuda.memory_allocated(0) / 1e9
    reserved = torch.cuda.memory_reserved(0) / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU 0: {allocated:.2f} GB allocated, {reserved:.2f} GB reserved, {total:.2f} GB total")
    
    # Verify model is on GPU
    model_device = next(qwen_model.parameters()).device
    print(f"  Model device: {model_device}")
    if model_device.type == 'cpu':
        print("  ⚠️ Moving model to GPU...")
        qwen_model = qwen_model.to(device)
        model_device = next(qwen_model.parameters()).device
        print(f"  ✓ Model now on {model_device}")


✓ GPU memory cleared
  GPU 0: 3.66 GB allocated, 3.67 GB reserved, 25.30 GB total
  Model device: cuda:0


## 10.5. Metrics Computation Functions


In [ ]:
# Function to extract ingredients and instructions from recipe text
def extract_recipe_sections(recipe_text):
    """Extract ingredients and instructions from recipe text."""
    recipe_lower = recipe_text.lower()
    
    # Try to find ingredients section
    ingredients = []
    if 'ingredients' in recipe_lower:
        try:
            ingredients_start = recipe_lower.find('ingredients')
            ingredients_end = recipe_lower.find('instructions', ingredients_start)
            if ingredients_end == -1:
                ingredients_end = recipe_lower.find('directions', ingredients_start)
            if ingredients_end == -1:
                ingredients_end = len(recipe_text)
            
            ingredients_section = recipe_text[ingredients_start:ingredients_end]
            # Extract lines with ingredients
            for line in ingredients_section.split('\n'):
                line = line.strip()
                if line and not line.lower().startswith('ingredients'):
                    # Remove markers like "-", "•", numbers
                    line = re.sub(r'^[-•\d.\s]+', '', line)
                    if line:
                        ingredients.append(line)
        except:
            pass
    
    # Try to find instructions section
    instructions = []
    if 'instructions' in recipe_lower or 'directions' in recipe_lower:
        try:
            instr_start = recipe_lower.find('instructions')
            if instr_start == -1:
                instr_start = recipe_lower.find('directions')
            
            instructions_section = recipe_text[instr_start:]
            # Extract lines with instructions
            for line in instructions_section.split('\n'):
                line = line.strip()
                if line and not line.lower().startswith(('instructions', 'directions')):
                    # Remove markers
                    line = re.sub(r'^[-•\d.\s]+', '', line)
                    if line:
                        instructions.append(line)
        except:
            pass
    
    return ingredients, instructions


def compute_f1_score(pred_items, gold_items):
    """Compute F1 score for sets of items (e.g., ingredients)."""
    if not pred_items and not gold_items:
        return 1.0
    if not pred_items or not gold_items:
        return 0.0
    
    # Convert to sets of words (case-insensitive)
    pred_words = set()
    gold_words = set()
    
    for item in pred_items:
        pred_words.update(word.lower() for word in item.split())
    for item in gold_items:
        gold_words.update(word.lower() for word in item.split())
    
    if not pred_words or not gold_words:
        return 0.0
    
    intersection = pred_words & gold_words
    if not intersection:
        return 0.0
    
    precision = len(intersection) / len(pred_words)
    recall = len(intersection) / len(gold_words)
    
    if precision + recall == 0:
        return 0.0
    
    f1 = 2 * (precision * recall) / (precision + recall)
    return f1


def compute_recipe_metrics(generated_text, ground_truth_text):
    """
    Compute ROUGE-L, BLEU, and F1 scores for recipe generation.
    
    Args:
        generated_text: Generated recipe text
        ground_truth_text: Ground truth recipe text
    
    Returns:
        dict with rouge_l, bleu, ingredient_f1, instruction_f1 scores
    """
    metrics = {}
    
    # ROUGE-L score
    try:
        rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
        rouge_scores = rouge.score(ground_truth_text, generated_text)
        metrics['rouge_l'] = rouge_scores['rougeL'].fmeasure
    except:
        metrics['rouge_l'] = 0.0
    
    # BLEU score
    try:
        bleu = sacrebleu.corpus_bleu(
            [generated_text],
            [[ground_truth_text]],
            force=False,
            lowercase=True,
            tokenize='intl'
        )
        metrics['bleu'] = bleu.score / 100.0  # Normalize to 0-1
    except:
        metrics['bleu'] = 0.0
    
    # F1 scores for ingredients and instructions
    gen_ingredients, gen_instructions = extract_recipe_sections(generated_text)
    gold_ingredients, gold_instructions = extract_recipe_sections(ground_truth_text)
    
    metrics['ingredient_f1'] = compute_f1_score(gen_ingredients, gold_ingredients)
    metrics['instruction_f1'] = compute_f1_score(gen_instructions, gold_instructions)
    
    return metrics


print("✓ Metrics computation functions defined!")
print("  - ROUGE-L: Overall recipe similarity")
print("  - BLEU: N-gram overlap")
print("  - F1 (Ingredients): Ingredient overlap")
print("  - F1 (Instructions): Instruction overlap")


✓ Metrics computation functions defined!
  - ROUGE-L: Overall recipe similarity
  - BLEU: N-gram overlap
  - F1 (Ingredients): Ingredient overlap
  - F1 (Instructions): Instruction overlap


## 11. Training

In [13]:
# # Create trainer
# trainer = Trainer(
#     model=qwen_model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=test_dataset,
#     data_collator=data_collator,
# )

# print("✓ Trainer created")
# print("\nStarting training...")
# print("=" * 60)

# # Train the model
# trainer.train()

# print("\n" + "=" * 60)
# print("✓ Training complete!")

# # Save final model
# final_model_dir = "./qwen_recipe_model_final"
# trainer.save_model(final_model_dir)
# qwen_processor.save_pretrained(final_model_dir)
# print(f"✓ Model saved to {final_model_dir}")

# # Store training history for later use
# training_history = trainer.state.log_history if hasattr(trainer.state, 'log_history') else []
# print(f"✓ Training history recorded ({len(training_history)} log entries)")

In [14]:
## 11.4. Recipe Generation Function (Required for Evaluation)

def generate_recipe(image_path, model, processor, max_new_tokens=2048):
    """Generate a recipe from a food image."""
    
    # Load image
    image = Image.open(image_path).convert('RGB')
    
    # Create prompt
    prompt_text = "Generate a detailed cooking recipe from this food image. Include ingredients, instructions, cooking times, and temperatures."
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt_text}
            ]
        }
    ]
    
    # Process inputs
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    )
    
    # Move to device
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Get token IDs
    pad_token_id = processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id
    eos_token_id = processor.tokenizer.eos_token_id
    
    # Generate
    model.eval()
    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=pad_token_id,
            eos_token_id=eos_token_id,
            repetition_penalty=1.1,
        )
    
    # Decode only the newly generated tokens
    input_length = inputs['input_ids'].shape[1]
    generated_tokens = generated_ids[0][input_length:]
    
    # Decode
    recipe = processor.decode(generated_tokens, skip_special_tokens=True)
    
    return {
        'recipe': recipe,
        'image_path': image_path
    }

print("✓ Recipe generation function defined!")


✓ Recipe generation function defined!


## 11.5. Evaluate Model and Compute Metrics


In [15]:
# Evaluate model on test set and compute ROUGE-L, BLEU, and F1 scores
print("=" * 80)
print("EVALUATING MODEL METRICS")
print("=" * 80)

if 'qwen_model' in globals() and 'test_data' in globals() and len(test_data) > 0:
    print(f"\nEvaluating on {min(50, len(test_data))} test samples...")
    print("This will generate recipes and compare with ground truth.")
    
    # Use a subset for faster evaluation
    eval_samples = test_data[:min(50, len(test_data))]
    
    all_metrics = {
        'rouge_l': [],
        'bleu': [],
        'ingredient_f1': [],
        'instruction_f1': []
    }
    
    print("\nGenerating recipes and computing metrics...")
    for i, sample in enumerate(eval_samples, 1):
        if i % 10 == 0:
            print(f"  Progress: {i}/{len(eval_samples)}")
        
        try:
            # Generate recipe
            result = generate_recipe(
                sample['image_path'],
                qwen_model,
                qwen_processor,
                max_new_tokens=1024
            )
            
            generated_recipe = result['recipe']
            ground_truth_recipe = sample['recipe']
            
            # Compute metrics
            metrics = compute_recipe_metrics(generated_recipe, ground_truth_recipe)
            
            # Store results
            all_metrics['rouge_l'].append(metrics['rouge_l'])
            all_metrics['bleu'].append(metrics['bleu'])
            all_metrics['ingredient_f1'].append(metrics['ingredient_f1'])
            all_metrics['instruction_f1'].append(metrics['instruction_f1'])
            
        except Exception as e:
            print(f"  ⚠️ Error evaluating sample {i}: {str(e)[:100]}")
            # Add zeros for failed samples
            for key in all_metrics:
                all_metrics[key].append(0.0)
    
    # Calculate aggregate statistics
    print("\n" + "=" * 80)
    print("METRICS SUMMARY")
    print("=" * 80)
    
    for metric_name, scores in all_metrics.items():
        if scores:
            mean_score = np.mean(scores)
            std_score = np.std(scores)
            median_score = np.median(scores)
            print(f"\n{metric_name.upper().replace('_', ' ')}:")
            print(f"  Mean:   {mean_score:.4f} ± {std_score:.4f}")
            print(f"  Median: {median_score:.4f}")
            print(f"  Min:    {np.min(scores):.4f}")
            print(f"  Max:    {np.max(scores):.4f}")
    
    # Save metrics to file
    metrics_file = "evaluation_metrics.json"
    with open(metrics_file, 'w') as f:
        json.dump({
            'aggregate': {k: {
                'mean': float(np.mean(v)),
                'std': float(np.std(v)),
                'median': float(np.median(v)),
                'min': float(np.min(v)),
                'max': float(np.max(v))
            } for k, v in all_metrics.items()},
            'individual': all_metrics
        }, f, indent=2)
    
    print(f"\n✓ Metrics saved to {metrics_file}")
    print("=" * 80)
    
else:
    print("⚠️ Cannot evaluate - model or test data not available")
    print("   Make sure training has completed and test_data exists")
    all_metrics = None


EVALUATING MODEL METRICS
⚠️ Cannot evaluate - model or test data not available
   Make sure training has completed and test_data exists


## 11.6. Visualize Metrics


In [ ]:
# Visualize evaluation metrics
if 'all_metrics' in globals() and all_metrics is not None and len(all_metrics['rouge_l']) > 0:
    print("Generating metrics visualizations...")
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Recipe Generation Quality Metrics', fontsize=16, fontweight='bold')
    
    # 1. Overall Metrics Bar Chart
    ax1 = axes[0, 0]
    metric_names = ['ROUGE-L', 'BLEU', 'Ingredient F1', 'Instruction F1']
    metric_values = [
        np.mean(all_metrics['rouge_l']),
        np.mean(all_metrics['bleu']),
        np.mean(all_metrics['ingredient_f1']),
        np.mean(all_metrics['instruction_f1'])
    ]
    metric_stds = [
        np.std(all_metrics['rouge_l']),
        np.std(all_metrics['bleu']),
        np.std(all_metrics['ingredient_f1']),
        np.std(all_metrics['instruction_f1'])
    ]
    
    bars = ax1.bar(metric_names, metric_values, yerr=metric_stds, capsize=5, 
                   alpha=0.8, color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12'], 
                   edgecolor='black', linewidth=1.5)
    ax1.set_ylabel('Score', fontsize=12)
    ax1.set_title('Average Metrics', fontsize=13, fontweight='bold')
    ax1.set_ylim(0, 1)
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar, val, std in zip(bars, metric_values, metric_stds):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + std,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # 2. Metrics Distribution (Box Plot)
    ax2 = axes[0, 1]
    box_data = [
        all_metrics['rouge_l'],
        all_metrics['bleu'],
        all_metrics['ingredient_f1'],
        all_metrics['instruction_f1']
    ]
    bp = ax2.boxplot(box_data, labels=metric_names, patch_artist=True)
    for patch, color in zip(bp['boxes'], ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax2.set_ylabel('Score', fontsize=12)
    ax2.set_title('Metrics Distribution', fontsize=13, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    
    # 3. Metrics Over Samples (Line Plot)
    ax3 = axes[1, 0]
    num_samples = len(all_metrics['rouge_l'])
    sample_indices = range(1, num_samples + 1)
    ax3.plot(sample_indices, all_metrics['rouge_l'], 'o-', label='ROUGE-L', alpha=0.7, linewidth=2)
    ax3.plot(sample_indices, all_metrics['bleu'], 's-', label='BLEU', alpha=0.7, linewidth=2)
    ax3.plot(sample_indices, all_metrics['ingredient_f1'], '^-', label='Ingredient F1', alpha=0.7, linewidth=2)
    ax3.plot(sample_indices, all_metrics['instruction_f1'], 'd-', label='Instruction F1', alpha=0.7, linewidth=2)
    ax3.set_xlabel('Sample Index', fontsize=12)
    ax3.set_ylabel('Score', fontsize=12)
    ax3.set_title('Metrics Across Samples', fontsize=13, fontweight='bold')
    ax3.legend(fontsize=10)
    ax3.grid(True, alpha=0.3)
    ax3.set_ylim(0, 1)
    
    # 4. Metrics Correlation Heatmap
    ax4 = axes[1, 1]
    metrics_df = pd.DataFrame(all_metrics)
    correlation_matrix = metrics_df.corr()
    im = ax4.imshow(correlation_matrix, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)
    ax4.set_xticks(range(len(metric_names)))
    ax4.set_yticks(range(len(metric_names)))
    ax4.set_xticklabels(metric_names, rotation=45, ha='right')
    ax4.set_yticklabels(metric_names)
    ax4.set_title('Metrics Correlation', fontsize=13, fontweight='bold')
    
    # Add correlation values to heatmap
    for i in range(len(metric_names)):
        for j in range(len(metric_names)):
            text = ax4.text(j, i, f'{correlation_matrix.iloc[i, j]:.2f}',
                          ha="center", va="center", color="black", fontweight='bold')
    
    plt.colorbar(im, ax=ax4, label='Correlation')
    plt.tight_layout()
    
    # Save the plot
    metrics_plot_file = "recipe_evaluation_metrics.png"
    plt.savefig(metrics_plot_file, dpi=150, bbox_inches='tight')
    print(f"✓ Metrics visualization saved to {metrics_plot_file}")
    plt.show()
    
    # Also create a summary table
    print("\n" + "=" * 80)
    print("METRICS SUMMARY TABLE")
    print("=" * 80)
    summary_df = pd.DataFrame({
        'Metric': metric_names,
        'Mean': [f"{v:.4f}" for v in metric_values],
        'Std': [f"{s:.4f}" for s in metric_stds],
        'Median': [f"{np.median(all_metrics[k]):.4f}" 
                   for k in ['rouge_l', 'bleu', 'ingredient_f1', 'instruction_f1']],
    })
    print(summary_df.to_string(index=False))
    print("=" * 80)
    
else:
    print("⚠️ No metrics available for visualization")
    print("   Please run the evaluation cell first (cell 28)")


⚠️ No metrics available for visualization
   Please run the evaluation cell first (cell 28)


## 12. Load Saved Model (if training was completed earlier)

In [ ]:
# Load saved model for inference
model_path = "./qwen_recipe_model_final"

if os.path.exists(model_path) and os.path.exists(os.path.join(model_path, "config.json")):
    print(f"Loading saved model from {model_path}...")
    
    if use_multi_gpu and num_gpus > 1:
        device_map = "balanced"
    else:
        device_map = "auto"
    
    qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_path,
        dtype=torch.float16,
        device_map=device_map,
        low_cpu_mem_usage=True,
    )
    
    qwen_processor = AutoProcessor.from_pretrained(model_path)
    qwen_model.eval()
    
    print("✓ Model loaded successfully!")
else:
    print("⚠️ Saved model not found. Using base model or run training first.")
    # Model should already be loaded from previous cells

Loading saved model from ./qwen_recipe_model_final...


Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.15s/it]


✓ Model loaded successfully!


## 13. Recipe Generation Function

In [ ]:
def generate_recipe(image_path, model, processor, max_new_tokens=2048):
    """Generate a recipe from a food image."""
    
    # Load image
    image = Image.open(image_path).convert('RGB')
    
    # Create prompt
    prompt_text = "Generate a detailed cooking recipe from this food image. Include ingredients, instructions, cooking times, and temperatures."
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt_text}
            ]
        }
    ]
    
    # Process inputs
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    )
    
    # Move to device
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Get token IDs
    pad_token_id = processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id
    eos_token_id = processor.tokenizer.eos_token_id
    
    # Generate
    model.eval()
    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=pad_token_id,
            eos_token_id=eos_token_id,
            repetition_penalty=1.1,
        )
    
    # Decode only the newly generated tokens
    input_length = inputs['input_ids'].shape[1]
    generated_tokens = generated_ids[0][input_length:]
    
    # Decode
    recipe = processor.decode(generated_tokens, skip_special_tokens=True)
    
    return {
        'recipe': recipe,
        'image_path': image_path
    }

print("✓ Recipe generation function defined!")

✓ Recipe generation function defined!


## 14. Test Recipe Generation

In [ ]:
# Test recipe generation on a few samples
if processed_data:
    print("Testing recipe generation...")
    print("=" * 60)
    
    test_samples = random.sample(processed_data, min(3, len(processed_data)))
    
    for i, sample in enumerate(test_samples, 1):
        print(f"\n{'='*60}")
        print(f"Test {i}/{len(test_samples)}")
        print(f"Original Recipe: {sample['title']}")
        print(f"Image: {sample['image_path']}")
        print(f"\n{'='*60}")
        
        try:
            result = generate_recipe(
                sample['image_path'],
                qwen_model,
                qwen_processor,
                max_new_tokens=1024
            )
            
            print("\nGenerated Recipe:")
            print(result['recipe'])
            
        except Exception as e:
            print(f"\n⚠️ Error generating recipe: {e}")
            import traceback
            traceback.print_exc()
            
else:
    print("⚠️ No processed data available for testing")

⚠️ No processed data available for testing


## 15. Visualization of Training Metrics

In [ ]:
# Plot training metrics if available
if 'trainer' in globals() and hasattr(trainer, 'state'):
    try:
        log_history = trainer.state.log_history
        
        train_losses = [entry['loss'] for entry in log_history if 'loss' in entry]
        eval_losses = [entry['eval_loss'] for entry in log_history if 'eval_loss' in entry]
        learning_rates = [entry['learning_rate'] for entry in log_history if 'learning_rate' in entry]
        
        if train_losses:
            fig, axes = plt.subplots(2, 2, figsize=(14, 10))
            fig.suptitle('Training Metrics', fontsize=16, fontweight='bold')
            
            # Training Loss
            steps = range(len(train_losses))
            axes[0, 0].plot(steps, train_losses, 'b-', label='Training Loss', linewidth=2)
            if eval_losses:
                eval_steps = [i for i, entry in enumerate(log_history) if 'eval_loss' in entry]
                if len(eval_steps) == len(eval_losses):
                    axes[0, 0].plot(eval_steps, eval_losses, 'r-', label='Eval Loss', linewidth=2, marker='o')
            axes[0, 0].set_xlabel('Training Steps')
            axes[0, 0].set_ylabel('Loss')
            axes[0, 0].set_title('Training and Evaluation Loss')
            axes[0, 0].legend()
            axes[0, 0].grid(True, alpha=0.3)
            
            # Perplexity
            perplexities = [np.exp(loss) for loss in train_losses]
            axes[0, 1].plot(steps, perplexities, 'g-', label='Perplexity', linewidth=2)
            axes[0, 1].set_xlabel('Training Steps')
            axes[0, 1].set_ylabel('Perplexity')
            axes[0, 1].set_title('Training Perplexity')
            axes[0, 1].legend()
            axes[0, 1].grid(True, alpha=0.3)
            
            # Learning Rate
            if learning_rates:
                lr_steps = [i for i, entry in enumerate(log_history) if 'learning_rate' in entry]
                if lr_steps:
                    axes[1, 0].plot(lr_steps, learning_rates, 'purple', label='Learning Rate', linewidth=2)
            axes[1, 0].set_xlabel('Training Steps')
            axes[1, 0].set_ylabel('Learning Rate')
            axes[1, 0].set_title('Learning Rate Schedule')
            axes[1, 0].legend()
            axes[1, 0].grid(True, alpha=0.3)
            
            # Loss Reduction
            if len(train_losses) > 1:
                initial_loss = train_losses[0]
                loss_reductions = [((initial_loss - loss) / initial_loss) * 100 for loss in train_losses]
                axes[1, 1].plot(steps, loss_reductions, 'orange', label='Loss Reduction %', linewidth=2, marker='s')
                axes[1, 1].set_xlabel('Training Steps')
                axes[1, 1].set_ylabel('Loss Reduction (%)')
                axes[1, 1].set_title('Training Progress')
                axes[1, 1].legend()
                axes[1, 1].grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.savefig('training_metrics.png', dpi=150, bbox_inches='tight')
            print("✓ Training plots saved to 'training_metrics.png'")
            plt.show()
        else:
            print("⚠️ No training history available")
    except Exception as e:
        print(f"⚠️ Could not plot metrics: {e}")
else:
    print("⚠️ No training history available (training not run yet or log file not found)")

⚠️ No training history available (training not run yet or log file not found)


## 16. Gradio Web Interface


In [ ]:
import gradio as gr

def gradio_generate_recipe(image):
    """
    Wrapper function for Gradio interface
    """
    if image is None:
        return "⚠️ Please upload an image."
    
    try:
        # Ensure PIL Image
        if not isinstance(image, Image.Image):
            image = Image.fromarray(image)
        
        if image.mode != 'RGB':
            image = image.convert('RGB')
        
        # Save temp image path for compatibility with generate_recipe function
        temp_path = "temp_food_image.jpg"
        image.save(temp_path)
        
        # Generate recipe using the defined function
        result = generate_recipe(
            temp_path,
            qwen_model,
            qwen_processor,
            max_new_tokens=2048
        )
        
        # Format output
        recipe_text = result.get('recipe', str(result)) if isinstance(result, dict) else result
        
        output = f"""{'=' * 60}
{recipe_text}
{'=' * 60}
NOTE: This recipe was generated by AI and may require refinement.
Please verify ingredients and instructions before cooking.
"""
        
        # Cleanup
        if os.path.exists(temp_path):
            os.remove(temp_path)
        
        return output
        
    except Exception as e:
        import traceback
        error_msg = f"❌ Error: {str(e)}"
        print(error_msg)
        print(traceback.format_exc())
        return error_msg


# Create Gradio interface
with gr.Blocks(title="Food Recipe Generator (Qwen2.5-VL-3B)", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🍽️ Food to Recipe Generator
    ### Powered by Qwen2.5-VL-3B-Instruct
    
    Upload a food image and get a detailed AI-generated recipe with:
    - 📋 Complete ingredient list
    - ⏱️ Preparation and cooking time estimates
    - 🌡️ Cooking temperature and methods
    - 📝 Step-by-step instructions
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(
                label="Upload Food Image",
                type="pil",
                height=400
            )
            generate_btn = gr.Button(
                "🚀 Generate Recipe",
                variant="primary",
                size="lg"
            )
            
            gr.Markdown("""
            ### Tips:
            - Upload clear food images
            - Works best with single dishes
            - Recipes are generated using fine-tuned Qwen2.5-VL-3B
            - Powered by state-of-the-art vision-language model
            """)
        
        with gr.Column(scale=1):
            recipe_output = gr.Textbox(
                label="Generated Recipe",
                lines=25,
                max_lines=30,
                placeholder="Your recipe will appear here...",
                show_copy_button=True
            )
    
    # Connect function
    generate_btn.click(
        fn=gradio_generate_recipe,
        inputs=image_input,
        outputs=recipe_output
    )
    
    image_input.upload(
        fn=gradio_generate_recipe,
        inputs=image_input,
        outputs=recipe_output
    )
    
    gr.Markdown("""
    ---
    **Note**: Recipes are AI-generated and may need refinement.
    
    Built with [Qwen2.5-VL-3B-Instruct](https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct) + [Gradio](https://gradio.app)
    """)

print("✓ Gradio interface created!")
print("\nTo launch the interface, run:")
print("  demo.launch(share=True)  # For public link")
print("  demo.launch()            # For local access")


✓ Gradio interface created!

To launch the interface, run:
  demo.launch(share=True)  # For public link
  demo.launch()            # For local access


### Launch the Web Interface


In [ ]:
# Launch Gradio interface
# Uncomment one of the options below:

# Option 1: Local access only (http://localhost:7860)
demo.launch(server_name="0.0.0.0", server_port=7860)

# Option 2: Create public link (share=True)
# demo.launch(share=True)

# Option 3: Both local and public
# demo.launch(server_name="0.0.0.0", server_port=7860, share=True)


* Running on local URL:  http://0.0.0.0:7860
* To create a public link, set `share=True` in `launch()`.
